# Prototype Computing the Jacobian with AD

This notebook implements a prototype for computing the derivatives of a square problem via automatic differentiation. See https://github.com/Pyomo/pyomo/issues/3564

In [1]:
import pyomo.environ as pyo
import idaes # load solvers

In [2]:
# Define a simple linear optimization problem
# This problem has no degrees of freedom and only linear equality constraints.
model = pyo.ConcreteModel()

# Define variables
model.x = pyo.Var(domain=pyo.NonNegativeReals)
model.y = pyo.Var(domain=pyo.NonNegativeReals)

# Define objective function
model.obj = pyo.Objective(expr=3 * model.x + 4 * model.y, sense=pyo.maximize)

# Define constraints
model.con1 = pyo.Constraint(expr=2 * model.x + model.y == 8)
model.con2 = pyo.Constraint(expr=model.x + 2 * model.y == 6)

# Solve the problem
solver = pyo.SolverFactory('ipopt')
solver.solve(model)

# Print results
print(f"x = {model.x.value}")
print(f"y = {model.y.value}")
print(f"Objective value = {model.obj()}")

x = 3.333333333333333
y = 1.3333333333333333
Objective value = 15.333333333333332


In [3]:
from pyomo.core.expr.calculus.diff_with_pyomo import reverse_sd
from pyomo.core.expr.visitor import identify_variables
from pyomo.common.collections import ComponentSet

# first sort the variables
var_set = ComponentSet()
for c in model.component_data_objects(pyo.Constraint, descend_into=True, active=True):
    for v in identify_variables(c.body, include_fixed=False):
        var_set.add(v)
var_list = list(var_set)

# now get the jacobian
# probably better to use some sparse matrix data structure
jac = []
for c in model.component_data_objects(pyo.Constraint, descend_into=True, active=True):
    row = []
    assert c.equality
    der_map = reverse_sd(c.body)
    for v in var_list:
        if v in der_map:
            row.append(der_map[v])
        else:
            row.append(0)
    jac.append(row)

In [4]:
# print the jacobian
for row_ndx, row in enumerate(jac):
    for col_ndx, entry in enumerate(row):
        print(f'({row_ndx},{col_ndx}): {str(entry)}')

(0,0): 2
(0,1): 1
(1,0): 1
(1,1): 2
